In [3]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import tqdm
import pickle as pic
from tqdm.notebook import tqdm

In [4]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

In [8]:
a=env.observation_space.n

In [9]:
b=env.action_space.n

In [10]:
def q_tab(a,b):
    qt=np.zeros((a,b))
    return qt

In [11]:
def gp(qt,state):
    act=np.argmax(qt[state][:])
    return act

In [28]:
def egp(qt,state,ep):
    c=random.uniform(0,1)
    if c>ep:
        act=gp(qt,state)
    else:
        act=env.action_space.sample()
    return act

In [41]:
eps=250000
lr=0.8
evaleps=100
env_id = "Taxi-v3"  # Name of the environment
max_steps = 99  # Max steps per episode
gamma = 0.9  # Discounting rate
eval_seed = []  # The evaluation seed of the environment

# Exploration parameters
max_epsilon = 1.0  # Exploration probability at start
min_epsilon = 0.05  # Minimum exploration probability
decay_rate = 0.0005 

In [42]:
env.reset()

(434, {'prob': 1.0, 'action_mask': array([0, 1, 1, 0, 0, 0], dtype=int8)})

In [43]:
def train(eps,min_epsilon, max_epsilon, decay_rate,env,max_steps,qt):
    for e in tqdm(range(eps)):
        epsi=min_epsilon+(max_epsilon-min_epsilon)*np.exp(-decay_rate*e)
        state,i=env.reset()
        s=0
        ter,trun=False,False
        for s in range(max_steps):
            act=egp(qt,state,e)
            news,rew,ter,trun,i=env.step(act)
            qt[state][act]=qt[state][act]+lr*(rew+gamma*np.max(qt[news]-qt[state][act]))
            if ter or trun:
                break
            state=news
    return qt

In [44]:
qtt=q_tab(a,b)

In [45]:
qtt=train(eps,min_epsilon, max_epsilon, decay_rate,env,max_steps,qtt)

  0%|          | 0/250000 [00:00<?, ?it/s]

In [46]:
qtt

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [10.        , 11.11111111, 10.        , 11.11111111, 12.22222222,
         1.11111111],
       [14.44444444, 15.55555556, 14.44444444, 15.55555556, 16.66666667,
         5.55555556],
       ...,
       [16.66666667, 17.77777778, 16.66666667, 15.55555556,  6.66666667,
         6.66666667],
       [12.22222222, 13.33333333, 12.22222222, 13.33333333,  2.22222222,
         2.22222222],
       [20.        , 18.88888889, 20.        , 21.11111111, 10.        ,
        10.        ]])

In [64]:
def evalu(env,max_steps,eps,qtt):
    erew=[]
    for e in tqdm(range(eps)):
        state,i=env.reset()
        s=0
        trun,term=False,False
        trew=0
        for s in range(max_steps):
            act=gp(qtt,state)
            news,rew,term,trun,i=env.step(act)
            trew+=rew
            if term or trun:
                break
            state=news
        erew.append(trew)
    mrew=np.mean(erew)
    srew=np.std(erew)
    return mrew,srew   

In [48]:
mean_reward, std_reward = eval(env, max_steps,eps,qtt)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

  0%|          | 0/250000 [00:00<?, ?it/s]

Mean_reward=7.93 +/- 2.59


In [56]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json

In [49]:
def record_video(env, Qtable, out_directory, fps=1):
    """
    Generate a replay video of the agent
    :param env
    :param Qtable: Qtable of our agent
    :param out_directory
    :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
    """
    images = []
    terminated = False
    truncated = False
    state, info = env.reset(seed=random.randint(0, 500))
    img = env.render()
    images.append(img)
    while not terminated or truncated:
        # Take the action (index) that have the maximum expected future reward given that state
        action = np.argmax(Qtable[state][:])
        state, reward, terminated, truncated, info = env.step(
            action
        )  # We directly put next_state = state for recording logic
        img = env.render()
        images.append(img)
    imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [69]:
def push_to_hub(repo_id, model, env, video_fps=1, local_repo_path="hub"):
    """
    Evaluate, Generate a video and Upload a model to Hugging Face Hub.
    This method does the complete pipeline:
    - It evaluates the model
    - It generates the model card
    - It generates a replay video of the agent
    - It pushes everything to the Hub

    :param repo_id: repo_id: id of the model repository from the Hugging Face Hub
    :param env
    :param video_fps: how many frame per seconds to record our video replay
    (with taxi-v3 and frozenlake-v1 we use 1)
    :param local_repo_path: where the local repository is
    """
    _, repo_name = repo_id.split("/")

    eval_env = env
    api = HfApi()

    # Step 1: Create the repo
    repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
    )

    # Step 2: Download files
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Step 3: Save the model
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False

    # Pickle the model
    with open((repo_local_path) / "q-learning.pkl", "wb") as f:
        pic.dump(model, f)

    # Step 4: Evaluate the model and build JSON with evaluation metrics
    mean_reward, std_reward = evalu(
        eval_env, model["max_steps"], model["n_eval_episodes"], model["qtable"]
    )

    evaluate_data = {
        "env_id": model["env_id"],
        "mean_reward": mean_reward,
        "n_eval_episodes": model["n_eval_episodes"],
        "eval_datetime": datetime.datetime.now().isoformat(),
    }

    # Write a JSON file called "results.json" that will contain the
    # evaluation results
    with open(repo_local_path / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")

    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-" + "no_slippery"

    metadata = {}
    metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

    # Add metrics
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
    )

    # Merges both dictionaries
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Q-Learning** Agent playing1 **{env_id}**
  This is a trained model of a **Q-Learning** agent playing **{env_id}** .

  ## Usage

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # Don't forget to check if you need to add additional attributes (is_slippery=False etc)
  env = gym.make(model["env_id"])
  """

    evalu(env, model["max_steps"], model["n_eval_episodes"], model["qtable"])

    readme_path = repo_local_path / "README.md"
    readme = ""
    print(readme_path.exists())
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)

    # Step 6: Record a video
    video_path = repo_local_path / "replay.mp4"
    record_video(env, model["qtable"], video_path, video_fps)

    # Step 7. Push everything to the Hub
    api.upload_folder(
        repo_id=repo_id,
        folder_path=repo_local_path,
        path_in_repo=".",
    )

    print("Your model is pushed to the Hub. You can view your model here: ", repo_url)

In [51]:
from huggingface_hub import notebook_login
notebook_login()

In [54]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": eps,
    "n_eval_episodes": evaleps,
    "learning_rate": lr,
    "gamma": gamma,
    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,
    "qtable": qtt,
}

In [70]:
username = "ByteMeHarder-404"
repo_name = "qlearning-taxi"
push_to_hub(repo_id=f"{username}/{repo_name}", model=model, env=env)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

False


error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (550, 350) to (560, 352) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_b

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Your model is pushed to the Hub. You can view your model here:  https://huggingface.co/ByteMeHarder-404/qlearning-taxi
